# 02 — Masking

**Purpose:** Apply point-source and cluster masks to the full-sky HEALPix CIB and tSZ maps.

This notebook covers both masking steps described in §2 of the paper:

1. **Point-source masking** — uses `get_point_source_mask_in_healpix` to identify pixels
   with flux above 2 mJy and sets them to zero. Note: the original `preprocessing.ipynb`
   used `astropy.stats.sigma_clip` instead; see `docs/paper_code_inconsistencies.md` for
   discussion.

2. **Cluster masking** — uses `get_apodised_mdpl2_cluster_mask` with the halo catalogue
   from `01_halo_catalogue.ipynb` to build apodised circular masks around clusters with
   M500c ≥ 3×10¹⁴ M☉, with radii of 3θ₅₀₀c–5θ₅₀₀c and a minimum of 10'. Masked pixels
   are inpainted with Gaussian random values matching the map mean and standard deviation.

**Inputs:**
- Full-sky CIB map: `data/agora_len_mag_cibmap_act_150ghz.fits`
- Full-sky tSZ map: `data/agora_ltszNG_bahamas80_bnd_unb_1.0e+12_1.0e+18_lensed.fits`
- Halo catalogue: `data/halo_catalogue/halo_catalogue_m500gt3e14.npy` (from `01_halo_catalogue.ipynb`)

**Outputs:**
- Masked CIB map: `data/cib_150_masked.fits`
- Masked tSZ map: `data/tsz_150_masked.fits`

**Key module functions:**
- `foregrounds_diffusion.get_cluster_source_mask_for_agora.get_point_source_mask_in_healpix`
- `foregrounds_diffusion.get_cluster_source_mask_for_agora.get_apodised_mdpl2_cluster_mask`

**Paper reference:** §2 (point-source and cluster masking descriptions).

In [2]:
!pip install -e ~/cmb_foregrounds_diffusion/foregrounds_diffusion

Obtaining file:///home/apb86/cmb_foregrounds_diffusion/foregrounds_diffusion

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: file:///home/apb86/cmb_foregrounds_diffusion/foregrounds_diffusion does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [5]:
import numpy as np
import healpy as hp

from foregrounds_diffusion.get_cluster_source_mask_for_agora import (
    get_point_source_mask_in_healpix,
    get_apodised_mdpl2_cluster_mask,
)

CIB_FITS  = "data/agora_len_mag_cibmap_act_150ghz.fits"
TSZ_FITS  = "data/agora_ltszNG_bahamas80_bnd_unb_1.0e+12_1.0e+18_lensed.fits"
HALO_CAT  = "data/halo_catalogue/halo_catalogue_m500gt3e14.npy"

FREQ              = 150.      # GHz
PTSRC_THRESH_MJY  = 2.0       # mJy threshold at 150 GHz
M500C_THRESHOLD   = 3e14      # M_sun


ModuleNotFoundError: No module named 'foregrounds_diffusion'

In [ ]:
cib_map = hp.read_map(CIB_FITS)
tsz_map = hp.read_map(TSZ_FITS)
nside   = hp.get_nside(cib_map)

print(f"NSIDE = {nside},  Npix = {hp.nside2npix(nside):,}")
print(f"CIB — min: {cib_map.min():.4f}  max: {cib_map.max():.4f}  std: {cib_map.std():.4f}")
print(f"tSZ — min: {tsz_map.min():.4f}  max: {tsz_map.max():.4f}  std: {tsz_map.std():.4f}")


In [ ]:
# Identify pixels with flux >= 2 mJy and zero them in both maps
ptsrc_pixels = get_point_source_mask_in_healpix(
    freq=FREQ,
    hmap_Mjy_per_sr=cib_map,
    threshold_mjy_freq0=PTSRC_THRESH_MJY,
    freq0=150.,
)
print(f"Point-source pixels above {PTSRC_THRESH_MJY} mJy: {len(ptsrc_pixels):,}")

cib_ptsrc = cib_map.copy()
tsz_ptsrc = tsz_map.copy()
cib_ptsrc[ptsrc_pixels] = 0.
tsz_ptsrc[ptsrc_pixels] = 0.


In [ ]:
# Build apodised cluster mask for halos with M500c >= 3e14 M_sun
cluster_mask = get_apodised_mdpl2_cluster_mask(
    nside=nside,
    halo_cat_fname=HALO_CAT,
    m500c_threshold=M500C_THRESHOLD,
    howmanythetaforclusters=3,   # mask radius = 3 × theta_500c
    apodise=True,
)
print(f"Cluster mask: sky fraction retained = {cluster_mask.mean():.3f}")

# Apply cluster mask; fill masked pixels with Gaussian noise
cib_masked = cib_ptsrc * cluster_mask
tsz_masked = tsz_ptsrc * cluster_mask

masked_idx = np.where(cluster_mask < 0.5)[0]
rng = np.random.default_rng(seed=0)
for hmap in (cib_masked, tsz_masked):
    unmasked = hmap[cluster_mask > 0.5]
    hmap[masked_idx] = rng.normal(loc=unmasked.mean(), scale=unmasked.std(),
                                  size=len(masked_idx))


In [ ]:
hp.write_map("data/cib_150_masked.fits", cib_masked, overwrite=True)
hp.write_map("data/tsz_150_masked.fits", tsz_masked, overwrite=True)
print("Saved masked maps.")
